# Chapter 6.10: Evaluation of Generative Recommendation

**Measuring what matters when recommendations are generated, not just retrieved.**

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Evaluate** generative recommendation systems beyond accuracy using diversity, novelty, and serendipity metrics
2. **Measure** explanation quality from LLM-based recommenders using BLEU, ROUGE, and BERTScore concepts
3. **Detect** hallucination in generative rec systems that recommend non-existent or inappropriate items
4. **Assess** fairness of generative recommendations across user groups and provider exposure
5. **Design** human evaluation protocols with pairwise comparisons and Likert scales
6. **Benchmark** generative vs discriminative rec systems and identify when each paradigm excels
7. **Build** a comprehensive evaluation framework combining automated metrics and hallucination detection

## Prerequisites

- Familiarity with standard recommendation metrics (Precision, Recall, NDCG)
- Understanding of generative recommendation models (Chapters 6.4–6.9)
- Basic knowledge of NLP evaluation (BLEU, ROUGE concepts)
- Python proficiency with NumPy and matplotlib

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part6/chapter_6.10_eval_generative.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part6/chapter_6.10_eval_generative.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print('Environment ready. All computations run on CPU with synthetic data.')

## 1. Beyond Accuracy: Why Traditional Metrics Fall Short

Traditional recommendation evaluation focuses on **accuracy**: did the system recommend items the user actually interacted with? Metrics like Precision@K, Recall@K, and NDCG@K dominate offline evaluation.

But generative recommendation introduces new capabilities and new failure modes:

| Dimension | Discriminative Rec | Generative Rec | New Challenge |
|-----------|-------------------|----------------|---------------|
| **Output** | Item IDs from catalog | Items + explanations + novel suggestions | Must evaluate text quality |
| **Item space** | Fixed catalog | Can "hallucinate" items | Must verify item existence |
| **Explanations** | Template-based | Free-form natural language | Must assess coherence & faithfulness |
| **Fairness** | Exposure-based | Generation bias amplification | Must audit language + exposure |
| **User trust** | Implicit (click/no-click) | Explicit (explanation quality) | Must run human evaluation |

> **💡 Concept:** A generative recommender that achieves perfect NDCG but always recommends popular items with hallucinated explanations is *worse* than a simpler system with honest, diverse suggestions. Evaluation must be multi-dimensional.

We organize evaluation into five pillars:

$$\text{Eval}_{\text{GenRec}} = \underbrace{\text{Accuracy}}_{\text{Relevance}} + \underbrace{\text{Beyond-Accuracy}}_{\text{Diversity, Novelty, Serendipity}} + \underbrace{\text{Text Quality}}_{\text{BLEU, ROUGE, BERTScore}} + \underbrace{\text{Safety}}_{\text{Hallucination, Fairness}} + \underbrace{\text{Human Judgment}}_{\text{Pairwise, Likert}}$$

In [ ]:
# ============================================================
# Synthetic Catalog & Recommendation Data
# ============================================================
# We create a realistic item catalog with categories, popularity,
# and provider information, then simulate outputs from two systems:
#   System A: a generative recommender (LLM-based)
#   System B: a discriminative recommender (traditional)

rng = np.random.RandomState(42)

N_ITEMS = 500
N_USERS = 200
K = 10  # recommendation list length

CATEGORIES = ['Electronics', 'Books', 'Clothing', 'Home', 'Sports',
              'Music', 'Food', 'Toys', 'Beauty', 'Garden']
PROVIDERS = [f'Provider_{i}' for i in range(50)]

# Item catalog
item_catalog = {
    'item_id': np.arange(N_ITEMS),
    'category': rng.choice(CATEGORIES, N_ITEMS),
    'provider': rng.choice(PROVIDERS, N_ITEMS),
    'popularity': rng.power(0.3, N_ITEMS),  # power-law popularity
}
item_catalog['popularity'] /= item_catalog['popularity'].sum()

# Ground-truth relevant items per user (simulate from latent factors)
user_factors = rng.randn(N_USERS, 16)
item_factors = rng.randn(N_ITEMS, 16)
relevance_scores = user_factors @ item_factors.T  # (N_USERS, N_ITEMS)

ground_truth = {}
for u in range(N_USERS):
    top_indices = np.argsort(relevance_scores[u])[::-1][:30]
    ground_truth[u] = set(top_indices.tolist())


def simulate_generative_recs(n_users, n_items, ground_truth, catalog, rng,
                              hallucination_rate=0.05):
    """Simulate a generative recommender that sometimes hallucinates."""
    recs = {}
    explanations = {}
    
    explanation_templates = [
        "Based on your interest in {cat}, you might enjoy this {cat2} item.",
        "Users with similar taste frequently purchased this alongside {cat} products.",
        "This is a highly rated {cat} item that matches your browsing history.",
        "Trending in {cat}: a unique pick that balances quality and value.",
        "A serendipitous find in {cat2} that complements your {cat} preferences.",
    ]
    
    for u in range(n_users):
        rec_list = []
        exp_list = []
        gt = ground_truth[u]
        
        for pos in range(K):
            if rng.random() < hallucination_rate:
                # Hallucinate: recommend a non-existent item ID
                fake_id = n_items + rng.randint(1, 1000)
                rec_list.append(fake_id)
                exp_list.append("A fantastic new product you will love!")
            else:
                # Mix of relevant and exploration
                if rng.random() < 0.5 and len(gt) > 0:
                    item = rng.choice(list(gt))
                else:
                    item = rng.choice(n_items)
                rec_list.append(item)
                cat = catalog['category'][item % n_items]
                cat2 = rng.choice(CATEGORIES)
                tmpl = rng.choice(explanation_templates)
                exp_list.append(tmpl.format(cat=cat, cat2=cat2))
        
        recs[u] = rec_list
        explanations[u] = exp_list
    
    return recs, explanations


def simulate_discriminative_recs(n_users, n_items, ground_truth, catalog, rng):
    """Simulate a discriminative recommender (no hallucination, popularity-biased)."""
    recs = {}
    for u in range(n_users):
        rec_list = []
        gt = ground_truth[u]
        for pos in range(K):
            if rng.random() < 0.6 and len(gt) > 0:
                item = rng.choice(list(gt))
            else:
                # Popularity-biased sampling
                item = rng.choice(n_items, p=catalog['popularity'])
            rec_list.append(item)
        recs[u] = rec_list
    return recs


gen_recs, gen_explanations = simulate_generative_recs(
    N_USERS, N_ITEMS, ground_truth, item_catalog, rng, hallucination_rate=0.05)
disc_recs = simulate_discriminative_recs(
    N_USERS, N_ITEMS, ground_truth, item_catalog, rng)

print(f'Catalog: {N_ITEMS} items, {len(CATEGORIES)} categories, {len(PROVIDERS)} providers')
print(f'Users: {N_USERS}, List size K={K}')
print(f'Sample generative rec for user 0: {gen_recs[0]}')
print(f'Sample explanation: "{gen_explanations[0][0]}"')
print(f'Sample discriminative rec for user 0: {disc_recs[0]}')

## 2. Diversity, Novelty, and Serendipity Metrics

### Intra-List Diversity (ILD)

Measures how different items within a single recommendation list are from each other:

$$\text{ILD}(L) = \frac{2}{|L|(|L|-1)} \sum_{i,j \in L, i \neq j} \text{dist}(i, j)$$

where $\text{dist}(i, j) = 1 - \mathbb{1}[\text{cat}(i) = \text{cat}(j)]$ for category-based diversity.

### Coverage

The fraction of the catalog that appears in recommendations across all users:

$$\text{Coverage} = \frac{|\bigcup_{u} L_u|}{|\mathcal{I}|}$$

### Novelty

Measures how unexpected the recommendations are, using item popularity as a proxy:

$$\text{Novelty}(L) = \frac{1}{|L|} \sum_{i \in L} -\log_2 p(i)$$

where $p(i)$ is the item's popularity. Recommending rare items yields higher novelty.

### Serendipity

Serendipity captures *unexpected relevance* -- items that are both surprising and useful:

$$\text{Serendipity}(L, u) = \frac{1}{|L|} \sum_{i \in L} \mathbb{1}[i \in \text{Rel}(u)] \cdot (1 - p(i))$$

> **💡 Concept:** A system that recommends obscure but irrelevant items has high novelty but low serendipity. Serendipity requires both surprise *and* relevance -- the "pleasant surprise" factor.

In [ ]:
# ============================================================
# Beyond-Accuracy Metrics Implementation
# ============================================================

def precision_at_k(rec_list, relevant_set, k=10):
    """Fraction of recommended items that are relevant."""
    hits = sum(1 for item in rec_list[:k] if item in relevant_set)
    return hits / k


def ndcg_at_k(rec_list, relevant_set, k=10):
    """Normalized Discounted Cumulative Gain."""
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(rec_list[:k])
             if item in relevant_set)
    ideal_hits = min(len(relevant_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def intra_list_diversity(rec_list, catalog, k=10):
    """Category-based intra-list diversity."""
    items = [i for i in rec_list[:k] if i < len(catalog['category'])]
    if len(items) < 2:
        return 0.0
    cats = [catalog['category'][i] for i in items]
    n_pairs = 0
    diversity_sum = 0.0
    for i in range(len(cats)):
        for j in range(i + 1, len(cats)):
            diversity_sum += (1.0 if cats[i] != cats[j] else 0.0)
            n_pairs += 1
    return diversity_sum / n_pairs if n_pairs > 0 else 0.0


def catalog_coverage(all_recs, n_items, k=10):
    """Fraction of catalog appearing in any user's list."""
    seen = set()
    for u, rec_list in all_recs.items():
        for item in rec_list[:k]:
            if item < n_items:
                seen.add(item)
    return len(seen) / n_items


def novelty(rec_list, popularity, k=10):
    """Average self-information of recommended items."""
    scores = []
    for item in rec_list[:k]:
        if item < len(popularity):
            p = max(popularity[item], 1e-10)
            scores.append(-np.log2(p))
    return np.mean(scores) if scores else 0.0


def serendipity(rec_list, relevant_set, popularity, k=10):
    """Unexpected relevance: relevant items that are also surprising."""
    scores = []
    for item in rec_list[:k]:
        if item in relevant_set and item < len(popularity):
            scores.append(1.0 - popularity[item])
        else:
            scores.append(0.0)
    return np.mean(scores)


# Compute all metrics for both systems
metrics = {'Generative': defaultdict(list), 'Discriminative': defaultdict(list)}

for u in range(N_USERS):
    gt = ground_truth[u]
    pop = item_catalog['popularity']
    
    for sys_name, recs in [('Generative', gen_recs), ('Discriminative', disc_recs)]:
        rec_list = recs[u]
        metrics[sys_name]['Precision@10'].append(precision_at_k(rec_list, gt))
        metrics[sys_name]['NDCG@10'].append(ndcg_at_k(rec_list, gt))
        metrics[sys_name]['ILD'].append(intra_list_diversity(rec_list, item_catalog))
        metrics[sys_name]['Novelty'].append(novelty(rec_list, pop))
        metrics[sys_name]['Serendipity'].append(serendipity(rec_list, gt, pop))

gen_coverage = catalog_coverage(gen_recs, N_ITEMS)
disc_coverage = catalog_coverage(disc_recs, N_ITEMS)

# Print summary
print(f'{"Metric":<18} {"Generative":>12} {"Discriminative":>14}')
print('-' * 46)
for m in ['Precision@10', 'NDCG@10', 'ILD', 'Novelty', 'Serendipity']:
    g = np.mean(metrics['Generative'][m])
    d = np.mean(metrics['Discriminative'][m])
    print(f'{m:<18} {g:>12.4f} {d:>14.4f}')
print(f'{"Coverage":<18} {gen_coverage:>12.4f} {disc_coverage:>14.4f}')

The metrics above give us the building blocks. Now let us visualize how the two systems compare across all dimensions simultaneously.

In [ ]:
# ============================================================
# Radar Chart: Multi-Dimensional Evaluation Comparison
# ============================================================

metric_names = ['Precision@10', 'NDCG@10', 'ILD', 'Novelty', 'Serendipity', 'Coverage']
gen_vals = [np.mean(metrics['Generative'][m]) for m in metric_names[:-1]] + [gen_coverage]
disc_vals = [np.mean(metrics['Discriminative'][m]) for m in metric_names[:-1]] + [disc_coverage]

# Normalize to [0, 1] for radar chart
max_vals = [max(g, d, 0.001) for g, d in zip(gen_vals, disc_vals)]
gen_norm = [g / m for g, m in zip(gen_vals, max_vals)]
disc_norm = [d / m for d, m in zip(disc_vals, max_vals)]

angles = np.linspace(0, 2 * np.pi, len(metric_names), endpoint=False).tolist()
gen_norm += gen_norm[:1]
disc_norm += disc_norm[:1]
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Radar chart
ax = fig.add_subplot(121, polar=True)
ax.plot(angles, gen_norm, 'o-', linewidth=2, label='Generative', color='crimson')
ax.fill(angles, gen_norm, alpha=0.15, color='crimson')
ax.plot(angles, disc_norm, 's-', linewidth=2, label='Discriminative', color='steelblue')
ax.fill(angles, disc_norm, alpha=0.15, color='steelblue')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_names, fontsize=9)
ax.set_title('Multi-Dimensional Evaluation\n(Normalized)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Remove the empty subplot created by subplots
axes[0].set_visible(False)
axes[1].set_visible(False)

# Bar chart comparison
ax2 = fig.add_subplot(122)
x = np.arange(len(metric_names))
width = 0.35
ax2.bar(x - width/2, gen_vals, width, label='Generative', color='crimson', alpha=0.8)
ax2.bar(x + width/2, disc_vals, width, label='Discriminative', color='steelblue', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(metric_names, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Score')
ax2.set_title('Generative vs Discriminative: Raw Scores', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Explanation Quality Metrics for LLM-Based Rec

When a generative recommender produces natural-language explanations, we need NLP metrics to assess quality.

### BLEU (Bilingual Evaluation Understudy)

Measures n-gram precision between generated and reference explanations:

$$\text{BLEU-N} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where $p_n$ is the modified n-gram precision and $\text{BP}$ is the brevity penalty:

$$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

Focuses on recall of n-grams:

$$\text{ROUGE-L} = \frac{(1 + \beta^2) \cdot R_{\text{lcs}} \cdot P_{\text{lcs}}}{R_{\text{lcs}} + \beta^2 \cdot P_{\text{lcs}}}$$

where $R_{\text{lcs}}$ and $P_{\text{lcs}}$ are based on the Longest Common Subsequence.

### BERTScore (Semantic Similarity)

Uses contextual embeddings to measure semantic similarity rather than exact n-gram overlap:

$$\text{BERTScore} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_i \in \hat{x}} \max_{x_j \in x} \cos(\mathbf{h}_{\hat{x}_i}, \mathbf{h}_{x_j})$$

> **⚠️ Common Pitfall:** BLEU penalizes creative but semantically correct explanations. A generative rec system might produce a novel but perfectly valid explanation that scores poorly on BLEU simply because it uses different words. Always complement n-gram metrics with semantic similarity (BERTScore) and human evaluation.

In [ ]:
# ============================================================
# Explanation Quality Metrics (Pure Python Implementations)
# ============================================================

def tokenize(text):
    """Simple whitespace + lowercase tokenizer."""
    return text.lower().strip().split()


def get_ngrams(tokens, n):
    """Extract n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def bleu_score(candidate, reference, max_n=4):
    """Compute BLEU score (simplified, single reference)."""
    cand_tokens = tokenize(candidate)
    ref_tokens = tokenize(reference)
    
    if len(cand_tokens) == 0:
        return 0.0
    
    # Brevity penalty
    bp = min(1.0, np.exp(1 - len(ref_tokens) / max(len(cand_tokens), 1)))
    
    log_avg = 0.0
    n_valid = 0
    for n in range(1, max_n + 1):
        cand_ngrams = get_ngrams(cand_tokens, n)
        ref_ngrams = get_ngrams(ref_tokens, n)
        if len(cand_ngrams) == 0:
            continue
        ref_counts = Counter(ref_ngrams)
        matches = sum(min(Counter(cand_ngrams)[ng], ref_counts[ng])
                      for ng in set(cand_ngrams))
        precision = matches / len(cand_ngrams)
        if precision > 0:
            log_avg += (1.0 / max_n) * np.log(precision)
            n_valid += 1
        else:
            log_avg += (1.0 / max_n) * np.log(1e-10)
            n_valid += 1
    
    return bp * np.exp(log_avg) if n_valid > 0 else 0.0


def lcs_length(x, y):
    """Longest Common Subsequence length."""
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def rouge_l(candidate, reference):
    """Compute ROUGE-L F1 score."""
    cand_tokens = tokenize(candidate)
    ref_tokens = tokenize(reference)
    if len(cand_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0
    lcs = lcs_length(cand_tokens, ref_tokens)
    precision = lcs / len(cand_tokens)
    recall = lcs / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def simulated_bertscore(candidate, reference, rng=None):
    """Simulated BERTScore using word overlap + noise.
    
    In production, this uses contextual embeddings (BERT/RoBERTa).
    Here we approximate with Jaccard similarity + random perturbation
    to demonstrate the evaluation pipeline.
    """
    if rng is None:
        rng = np.random.RandomState(0)
    cand_set = set(tokenize(candidate))
    ref_set = set(tokenize(reference))
    if len(cand_set) == 0 or len(ref_set) == 0:
        return 0.0
    jaccard = len(cand_set & ref_set) / len(cand_set | ref_set)
    # Add noise to simulate embedding-based scoring
    noise = rng.normal(0, 0.05)
    return np.clip(jaccard + noise + 0.3, 0.0, 1.0)  # bias upward like real BERTScore


# Generate reference explanations (what a "perfect" system would say)
reference_templates = [
    "Based on your interest in {cat}, this {cat} item is a great match.",
    "Users who liked {cat} products also enjoyed this item.",
    "This highly rated {cat} product aligns with your preferences.",
    "A popular choice in {cat} that fits your browsing pattern.",
    "Recommended for you based on your {cat} purchase history.",
]

bleu_scores = []
rouge_scores = []
bert_scores = []
eval_rng = np.random.RandomState(123)

for u in range(min(N_USERS, 100)):
    for pos in range(K):
        candidate = gen_explanations[u][pos]
        item = gen_recs[u][pos]
        if item < N_ITEMS:
            cat = item_catalog['category'][item]
        else:
            cat = 'Unknown'
        ref = eval_rng.choice(reference_templates).format(cat=cat)
        
        bleu_scores.append(bleu_score(candidate, ref, max_n=2))
        rouge_scores.append(rouge_l(candidate, ref))
        bert_scores.append(simulated_bertscore(candidate, ref, eval_rng))

print(f'Explanation Quality Metrics (averaged over {len(bleu_scores)} explanations):')
print(f'  BLEU-2:             {np.mean(bleu_scores):.4f} (std: {np.std(bleu_scores):.4f})')
print(f'  ROUGE-L:            {np.mean(rouge_scores):.4f} (std: {np.std(rouge_scores):.4f})')
print(f'  BERTScore (approx): {np.mean(bert_scores):.4f} (std: {np.std(bert_scores):.4f})')

## 4. Hallucination in Generative Recommendation

One of the most critical risks of generative recommendation is **hallucination**: the system recommends items that do not exist in the catalog, fabricates product attributes, or generates misleading explanations.

### Types of Hallucination in GenRec

| Type | Description | Example | Severity |
|------|------------|---------|----------|
| **Item hallucination** | Recommends non-existent item IDs | "Try item #99999" (not in catalog) | Critical |
| **Attribute hallucination** | Fabricates item properties | "This laptop has 128GB RAM" (it has 16GB) | High |
| **Explanation hallucination** | Generates unfaithful reasoning | "You bought X" (user never bought X) | Medium |
| **Popularity hallucination** | Falsely claims popularity | "Best seller in category" (actually unpopular) | Medium |

### Detection Methods

1. **Catalog verification**: Check if recommended item IDs exist in the catalog
2. **Attribute cross-referencing**: Verify generated attributes against the item database
3. **Consistency checking**: Compare explanations against user history
4. **Confidence calibration**: Flag low-confidence generations

> **⚠️ Common Pitfall:** Hallucination rates often increase for long-tail items where the LLM has less training data. Always stratify hallucination analysis by item popularity.

In [ ]:
# ============================================================
# Hallucination Detection Framework
# ============================================================

def detect_item_hallucination(rec_list, n_catalog_items):
    """Check if any recommended items are outside the catalog."""
    hallucinated = [item for item in rec_list if item >= n_catalog_items]
    return hallucinated


def detect_explanation_hallucination(explanation, user_history_cats, item_cat):
    """Check if explanation references categories not in user history.
    
    Returns a hallucination score [0, 1] based on ungrounded claims.
    """
    tokens = tokenize(explanation)
    mentioned_cats = [cat.lower() for cat in CATEGORIES
                      if cat.lower() in explanation.lower()]
    
    if not mentioned_cats:
        return 0.0  # No category claims to verify
    
    user_cats_lower = {c.lower() for c in user_history_cats}
    ungrounded = [c for c in mentioned_cats
                  if c not in user_cats_lower and c != item_cat.lower()]
    
    return len(ungrounded) / max(len(mentioned_cats), 1)


def confidence_based_detection(rec_scores, threshold=0.1):
    """Flag recommendations with low generation confidence."""
    # Simulate confidence scores (in practice, from model logits)
    return [i for i, s in enumerate(rec_scores) if s < threshold]


# Simulate user histories for hallucination checking
user_history_cats = {}
for u in range(N_USERS):
    # Each user has interacted with items from a subset of categories
    n_cats = rng.randint(2, 6)
    user_history_cats[u] = set(rng.choice(CATEGORIES, n_cats, replace=False))

# Run hallucination detection
item_hallucination_counts = []
explanation_hallucination_scores = []

for u in range(N_USERS):
    # Item hallucination
    hallucinated = detect_item_hallucination(gen_recs[u], N_ITEMS)
    item_hallucination_counts.append(len(hallucinated))
    
    # Explanation hallucination
    for pos in range(K):
        item = gen_recs[u][pos]
        if item < N_ITEMS:
            item_cat = item_catalog['category'][item]
        else:
            item_cat = 'Unknown'
        score = detect_explanation_hallucination(
            gen_explanations[u][pos], user_history_cats[u], item_cat)
        explanation_hallucination_scores.append(score)

total_recs = N_USERS * K
total_item_hallucinations = sum(item_hallucination_counts)
users_with_hallucination = sum(1 for c in item_hallucination_counts if c > 0)

print('=== Hallucination Detection Report ===')
print(f'Total recommendations analyzed: {total_recs}')
print(f'\nItem Hallucination:')
print(f'  Hallucinated items: {total_item_hallucinations} '
      f'({100*total_item_hallucinations/total_recs:.1f}%)')
print(f'  Users affected: {users_with_hallucination}/{N_USERS} '
      f'({100*users_with_hallucination/N_USERS:.1f}%)')
print(f'\nExplanation Hallucination:')
print(f'  Mean hallucination score: {np.mean(explanation_hallucination_scores):.4f}')
print(f'  Explanations with any hallucination: '
      f'{sum(1 for s in explanation_hallucination_scores if s > 0)} / '
      f'{len(explanation_hallucination_scores)}')

## 5. Fairness Evaluation for Generative Rec

Generative recommendation raises unique fairness concerns:

### Group Fairness (User-Side)

Are recommendations equally good for different user groups?

$$\Delta_{\text{fair}} = |\text{NDCG}(G_1) - \text{NDCG}(G_2)|$$

### Provider Exposure Fairness

Do all providers get fair exposure in recommendations? We use the Gini coefficient:

$$\text{Gini} = \frac{\sum_{i=1}^{n} \sum_{j=1}^{n} |x_i - x_j|}{2n \sum_{i=1}^{n} x_i}$$

where $x_i$ is the exposure count for provider $i$. $\text{Gini} = 0$ means perfect equality; $\text{Gini} = 1$ means one provider gets all exposure.

### Language Bias in Explanations

Generative systems can introduce language bias:
- More enthusiastic language for popular items
- Different sentiment across categories
- Stereotypical associations

> **🔑 Pro Tip:** Always disaggregate metrics by user demographics and item categories. Aggregate metrics can mask significant disparities.

In [ ]:
# ============================================================
# Fairness Evaluation
# ============================================================

def gini_coefficient(values):
    """Compute Gini coefficient for exposure distribution."""
    values = np.array(values, dtype=float)
    if len(values) == 0 or values.sum() == 0:
        return 0.0
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals) / (n * np.sum(sorted_vals))) - (n + 1) / n


def provider_exposure(all_recs, catalog, n_items):
    """Count how many times each provider's items appear in recs."""
    exposure = Counter()
    for u, rec_list in all_recs.items():
        for item in rec_list:
            if item < n_items:
                provider = catalog['provider'][item]
                exposure[provider] += 1
    return exposure


# Assign users to groups (simulate demographics)
user_groups = {}
for u in range(N_USERS):
    user_groups[u] = 'Group_A' if rng.random() < 0.6 else 'Group_B'

# Group fairness: NDCG gap between groups
group_metrics = defaultdict(lambda: defaultdict(list))

for u in range(N_USERS):
    group = user_groups[u]
    gt = ground_truth[u]
    for sys_name, recs in [('Generative', gen_recs), ('Discriminative', disc_recs)]:
        ndcg = ndcg_at_k(recs[u], gt)
        group_metrics[sys_name][group].append(ndcg)

print('=== Group Fairness (NDCG@10 by User Group) ===')
for sys_name in ['Generative', 'Discriminative']:
    ga = np.mean(group_metrics[sys_name]['Group_A'])
    gb = np.mean(group_metrics[sys_name]['Group_B'])
    gap = abs(ga - gb)
    print(f'  {sys_name:15s}: Group_A={ga:.4f}, Group_B={gb:.4f}, Gap={gap:.4f}')

# Provider exposure fairness
gen_exposure = provider_exposure(gen_recs, item_catalog, N_ITEMS)
disc_exposure = provider_exposure(disc_recs, item_catalog, N_ITEMS)

# Ensure all providers are counted (even with 0 exposure)
gen_exp_vals = [gen_exposure.get(p, 0) for p in PROVIDERS]
disc_exp_vals = [disc_exposure.get(p, 0) for p in PROVIDERS]

gen_gini = gini_coefficient(gen_exp_vals)
disc_gini = gini_coefficient(disc_exp_vals)

print(f'\n=== Provider Exposure Fairness ===')
print(f'  Generative    Gini: {gen_gini:.4f}')
print(f'  Discriminative Gini: {disc_gini:.4f}')
print(f'  (Lower Gini = more equitable exposure)')

# Plot provider exposure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sorted_gen = sorted(gen_exp_vals, reverse=True)
sorted_disc = sorted(disc_exp_vals, reverse=True)

axes[0].bar(range(len(PROVIDERS)), sorted_gen, color='crimson', alpha=0.7, label='Generative')
axes[0].bar(range(len(PROVIDERS)), sorted_disc, color='steelblue', alpha=0.5, label='Discriminative')
axes[0].set_xlabel('Provider Rank')
axes[0].set_ylabel('Exposure Count')
axes[0].set_title(f'Provider Exposure Distribution\n'
                   f'Gini: Gen={gen_gini:.3f}, Disc={disc_gini:.3f}', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Lorenz curve
def lorenz_curve(values):
    sorted_v = np.sort(values)
    cumulative = np.cumsum(sorted_v) / np.sum(sorted_v)
    return np.concatenate([[0], cumulative])

x_axis = np.linspace(0, 1, len(PROVIDERS) + 1)
axes[1].plot(x_axis, lorenz_curve(gen_exp_vals), 'r-o', markersize=3,
             label=f'Generative (Gini={gen_gini:.3f})', linewidth=2)
axes[1].plot(x_axis, lorenz_curve(disc_exp_vals), 'b-s', markersize=3,
             label=f'Discriminative (Gini={disc_gini:.3f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Equality')
axes[1].set_xlabel('Cumulative Share of Providers')
axes[1].set_ylabel('Cumulative Share of Exposure')
axes[1].set_title('Lorenz Curve: Provider Exposure Fairness', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Human Evaluation Protocols

Automated metrics capture only part of the picture. Human evaluation remains essential for generative recommendation.

### Pairwise Comparison (A/B Style)

Show evaluators two recommendations (from System A and System B) and ask:
- *"Which recommendation is more relevant?"*
- *"Which explanation is more helpful?"*
- *"Which set would you prefer to receive?"*

The win rate is computed as:

$$\text{Win Rate}_A = \frac{\text{\# times A preferred}}{\text{\# comparisons}}$$

### Likert Scale Evaluation

Rate individual aspects on a 1-5 scale:

| Score | Meaning |
|-------|--------|
| 5 | Excellent -- highly relevant and well-explained |
| 4 | Good -- relevant with minor issues |
| 3 | Acceptable -- somewhat relevant |
| 2 | Poor -- mostly irrelevant or misleading |
| 1 | Unacceptable -- irrelevant, hallucinated, or harmful |

### Inter-Annotator Agreement

Use Cohen's Kappa to measure agreement:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is observed agreement and $p_e$ is expected agreement by chance.

> **🔑 Pro Tip:** For reliable human evaluation, use at least 3 annotators per sample, randomize system order to avoid position bias, and collect at least 200-500 comparisons for statistical significance.

In [ ]:
# ============================================================
# Simulated Human Evaluation
# ============================================================

def simulate_pairwise_comparison(gen_recs, disc_recs, ground_truth,
                                  n_comparisons=200, rng=None):
    """Simulate human pairwise preferences.
    
    Simulates that humans tend to prefer:
    - More relevant recommendations
    - More diverse recommendations
    - Better-explained recommendations (generative advantage)
    """
    if rng is None:
        rng = np.random.RandomState(42)
    
    results = {'gen_wins': 0, 'disc_wins': 0, 'ties': 0}
    
    users = rng.choice(N_USERS, n_comparisons, replace=True)
    for u in users:
        gt = ground_truth[u]
        gen_rel = sum(1 for i in gen_recs[u] if i in gt)
        disc_rel = sum(1 for i in disc_recs[u] if i in gt)
        
        # Humans also value diversity (slight bonus for generative)
        gen_div = intra_list_diversity(gen_recs[u], item_catalog)
        disc_div = intra_list_diversity(disc_recs[u], item_catalog)
        
        gen_score = gen_rel + 0.5 * gen_div + rng.normal(0, 0.5)
        disc_score = disc_rel + 0.5 * disc_div + rng.normal(0, 0.5)
        
        if gen_score > disc_score + 0.3:
            results['gen_wins'] += 1
        elif disc_score > gen_score + 0.3:
            results['disc_wins'] += 1
        else:
            results['ties'] += 1
    
    return results


def simulate_likert_ratings(recs, ground_truth, catalog, n_items,
                             n_eval=100, rng=None):
    """Simulate Likert-scale ratings (1-5) for recommendations."""
    if rng is None:
        rng = np.random.RandomState(42)
    
    ratings = {'relevance': [], 'diversity': [], 'explanation': [], 'trust': []}
    
    users = rng.choice(N_USERS, n_eval, replace=False)
    for u in users:
        gt = ground_truth[u]
        rec_list = recs[u]
        
        # Relevance rating
        rel_frac = sum(1 for i in rec_list if i in gt) / K
        rel_rating = np.clip(int(1 + 4 * rel_frac + rng.normal(0, 0.5)), 1, 5)
        ratings['relevance'].append(rel_rating)
        
        # Diversity rating
        div = intra_list_diversity(rec_list, catalog)
        div_rating = np.clip(int(1 + 4 * div + rng.normal(0, 0.5)), 1, 5)
        ratings['diversity'].append(div_rating)
        
        # Explanation rating (only for generative)
        exp_rating = np.clip(int(3 + rng.normal(0.5, 0.8)), 1, 5)
        ratings['explanation'].append(exp_rating)
        
        # Trust rating (penalized by hallucination)
        n_halluc = len(detect_item_hallucination(rec_list, n_items))
        trust_base = 4 if n_halluc == 0 else max(1, 4 - 2 * n_halluc)
        trust_rating = np.clip(int(trust_base + rng.normal(0, 0.5)), 1, 5)
        ratings['trust'].append(trust_rating)
    
    return ratings


def cohens_kappa(ratings1, ratings2):
    """Compute Cohen's Kappa for inter-annotator agreement."""
    assert len(ratings1) == len(ratings2)
    n = len(ratings1)
    po = sum(1 for a, b in zip(ratings1, ratings2) if a == b) / n
    
    # Expected agreement
    counter1 = Counter(ratings1)
    counter2 = Counter(ratings2)
    all_labels = set(ratings1) | set(ratings2)
    pe = sum((counter1[l] / n) * (counter2[l] / n) for l in all_labels)
    
    if pe == 1.0:
        return 1.0
    return (po - pe) / (1 - pe)


# Run pairwise comparison
pairwise = simulate_pairwise_comparison(gen_recs, disc_recs, ground_truth)
total = sum(pairwise.values())
print('=== Pairwise Comparison Results ===')
print(f'  Generative wins:     {pairwise["gen_wins"]:3d} ({100*pairwise["gen_wins"]/total:.1f}%)')
print(f'  Discriminative wins: {pairwise["disc_wins"]:3d} ({100*pairwise["disc_wins"]/total:.1f}%)')
print(f'  Ties:                {pairwise["ties"]:3d} ({100*pairwise["ties"]/total:.1f}%)')

# Run Likert evaluation
likert = simulate_likert_ratings(gen_recs, ground_truth, item_catalog, N_ITEMS,
                                  n_eval=100, rng=np.random.RandomState(99))
print(f'\n=== Likert Scale Ratings (Generative System) ===')
for dim, vals in likert.items():
    print(f'  {dim.capitalize():13s}: mean={np.mean(vals):.2f}, '
          f'median={np.median(vals):.0f}, std={np.std(vals):.2f}')

# Inter-annotator agreement (simulate 2 annotators)
rng2 = np.random.RandomState(77)
likert2 = simulate_likert_ratings(gen_recs, ground_truth, item_catalog, N_ITEMS,
                                   n_eval=100, rng=rng2)
kappa = cohens_kappa(likert['relevance'], likert2['relevance'])
print(f'\n=== Inter-Annotator Agreement ===')
print(f'  Cohen\'s Kappa (relevance): {kappa:.4f}')

## 7. Benchmarking Generative vs Discriminative: When Does Generative Win?

Generative recommendation does not uniformly dominate discriminative approaches. The right choice depends on the scenario:

| Scenario | Winner | Why |
|----------|--------|-----|
| **Cold-start items** | Generative | Can leverage item descriptions/metadata via LLM |
| **Explanation required** | Generative | Produces natural-language rationale |
| **Large catalog, dense interactions** | Discriminative | More efficient, less hallucination risk |
| **Latency-critical** | Discriminative | Lighter inference, no autoregressive decoding |
| **Diverse exploration** | Generative | Natural stochasticity in generation |
| **Strict catalog compliance** | Discriminative | Zero hallucination by construction |
| **Cross-domain** | Generative | LLM knowledge transfer across domains |

### Evaluation Protocol for Fair Comparison

1. **Same data split** -- identical train/val/test sets
2. **Same metric suite** -- accuracy + beyond-accuracy + fairness
3. **Computational budget** -- normalize by FLOPs or wall-clock time
4. **Statistical testing** -- paired t-test or bootstrap confidence intervals

In [ ]:
# ============================================================
# Scenario-Based Benchmarking
# ============================================================

def bootstrap_confidence_interval(values, n_bootstrap=1000, ci=0.95, rng=None):
    """Compute bootstrap confidence interval for the mean."""
    if rng is None:
        rng = np.random.RandomState(42)
    means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        means.append(np.mean(sample))
    lower = np.percentile(means, (1 - ci) / 2 * 100)
    upper = np.percentile(means, (1 + ci) / 2 * 100)
    return np.mean(values), lower, upper


# Simulate scenario-specific performance
scenarios = {
    'Cold-Start Items': {'gen_boost': 0.15, 'disc_boost': 0.0},
    'Dense Interactions': {'gen_boost': 0.0, 'disc_boost': 0.10},
    'Explanation Needed': {'gen_boost': 0.20, 'disc_boost': 0.0},
    'Latency-Critical': {'gen_boost': -0.05, 'disc_boost': 0.12},
    'Cross-Domain': {'gen_boost': 0.18, 'disc_boost': 0.02},
    'Catalog Compliance': {'gen_boost': -0.10, 'disc_boost': 0.15},
}

base_gen_ndcg = np.mean(metrics['Generative']['NDCG@10'])
base_disc_ndcg = np.mean(metrics['Discriminative']['NDCG@10'])

scenario_results = {}
for scenario, boosts in scenarios.items():
    gen_ndcg = base_gen_ndcg + boosts['gen_boost']
    disc_ndcg = base_disc_ndcg + boosts['disc_boost']
    scenario_results[scenario] = {
        'Generative': np.clip(gen_ndcg, 0, 1),
        'Discriminative': np.clip(disc_ndcg, 0, 1)
    }

# Bootstrap CI for overall comparison
gen_mean, gen_lo, gen_hi = bootstrap_confidence_interval(
    metrics['Generative']['NDCG@10'], rng=np.random.RandomState(42))
disc_mean, disc_lo, disc_hi = bootstrap_confidence_interval(
    metrics['Discriminative']['NDCG@10'], rng=np.random.RandomState(42))

print('=== Overall NDCG@10 with 95% Bootstrap CI ===')
print(f'  Generative:     {gen_mean:.4f} [{gen_lo:.4f}, {gen_hi:.4f}]')
print(f'  Discriminative: {disc_mean:.4f} [{disc_lo:.4f}, {disc_hi:.4f}]')

# Scenario comparison chart
fig, ax = plt.subplots(figsize=(10, 6))
scenario_names = list(scenario_results.keys())
gen_scores = [scenario_results[s]['Generative'] for s in scenario_names]
disc_scores = [scenario_results[s]['Discriminative'] for s in scenario_names]

y = np.arange(len(scenario_names))
height = 0.35

bars1 = ax.barh(y - height/2, gen_scores, height, label='Generative',
                color='crimson', alpha=0.8)
bars2 = ax.barh(y + height/2, disc_scores, height, label='Discriminative',
                color='steelblue', alpha=0.8)

# Mark the winner for each scenario
for i, (g, d) in enumerate(zip(gen_scores, disc_scores)):
    winner_x = max(g, d) + 0.01
    winner_label = 'G' if g > d else 'D'
    ax.text(winner_x, i, f'<- {winner_label} wins', va='center', fontsize=9,
            color='crimson' if g > d else 'steelblue', fontweight='bold')

ax.set_yticks(y)
ax.set_yticklabels(scenario_names)
ax.set_xlabel('NDCG@10')
ax.set_title('When Does Generative Win? Scenario-Based Comparison', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
ax.set_xlim(0, max(max(gen_scores), max(disc_scores)) + 0.15)

plt.tight_layout()
plt.show()

## 8. Putting It All Together: Evaluation Dashboard

A production evaluation framework should combine all dimensions into a single dashboard. Let us aggregate the results.

> **💡 Concept:** No single metric tells the full story. A comprehensive evaluation framework must balance accuracy, diversity, safety, fairness, and user satisfaction. The weights between these dimensions depend on your application context.

In [ ]:
# ============================================================
# Comprehensive Evaluation Dashboard
# ============================================================

dashboard = {
    'Accuracy': {
        'Precision@10': (np.mean(metrics['Generative']['Precision@10']),
                         np.mean(metrics['Discriminative']['Precision@10'])),
        'NDCG@10': (np.mean(metrics['Generative']['NDCG@10']),
                    np.mean(metrics['Discriminative']['NDCG@10'])),
    },
    'Beyond-Accuracy': {
        'ILD': (np.mean(metrics['Generative']['ILD']),
                np.mean(metrics['Discriminative']['ILD'])),
        'Novelty': (np.mean(metrics['Generative']['Novelty']),
                    np.mean(metrics['Discriminative']['Novelty'])),
        'Serendipity': (np.mean(metrics['Generative']['Serendipity']),
                        np.mean(metrics['Discriminative']['Serendipity'])),
        'Coverage': (gen_coverage, disc_coverage),
    },
    'Safety': {
        'Hallucination Rate': (total_item_hallucinations / total_recs, 0.0),
        'Explanation Faithfulness': (1 - np.mean(explanation_hallucination_scores), 1.0),
    },
    'Fairness': {
        'Group NDCG Gap': (
            abs(np.mean(group_metrics['Generative']['Group_A']) -
                np.mean(group_metrics['Generative']['Group_B'])),
            abs(np.mean(group_metrics['Discriminative']['Group_A']) -
                np.mean(group_metrics['Discriminative']['Group_B']))
        ),
        'Provider Gini': (gen_gini, disc_gini),
    },
    'Human Eval': {
        'Win Rate': (pairwise['gen_wins'] / total, pairwise['disc_wins'] / total),
        'Avg Relevance Likert': (np.mean(likert['relevance']), float('nan')),
        'Avg Trust Likert': (np.mean(likert['trust']), float('nan')),
    },
}

print('=' * 70)
print(f'{"COMPREHENSIVE EVALUATION DASHBOARD":^70}')
print('=' * 70)

for category, metric_dict in dashboard.items():
    print(f'\n--- {category} ---')
    for metric_name, (gen_val, disc_val) in metric_dict.items():
        gen_str = f'{gen_val:.4f}'
        disc_str = f'{disc_val:.4f}' if not np.isnan(disc_val) else 'N/A'
        print(f'  {metric_name:28s}  Gen: {gen_str:>8}  Disc: {disc_str:>8}')

print('\n' + '=' * 70)

## Exercises

### Exercise 1: Implement a Comprehensive Evaluation Framework

Build a single `EvaluationFramework` class that computes all metrics (accuracy, diversity, novelty, hallucination detection) in one pass over the recommendations.

In [ ]:
class EvaluationFramework:
    """Comprehensive evaluation framework for generative recommendation.
    
    TODO: Implement the following methods:
    
    1. __init__: Store the catalog, ground truth, and configuration
    2. evaluate_accuracy: Compute Precision@K, Recall@K, NDCG@K
    3. evaluate_beyond_accuracy: Compute ILD, Coverage, Novelty, Serendipity
    4. evaluate_hallucination: Detect item and explanation hallucinations
    5. evaluate_fairness: Compute group fairness gap and provider Gini
    6. evaluate_all: Run all evaluations and return a consolidated report dict
    7. compare_systems: Given two sets of recs, produce a side-by-side report
    """
    
    def __init__(self, catalog, ground_truth, n_items, k=10):
        # TODO: Store parameters
        self.catalog = catalog
        self.ground_truth = ground_truth
        self.n_items = n_items
        self.k = k
    
    def evaluate_accuracy(self, recs):
        # TODO: Implement Precision@K, Recall@K, NDCG@K
        # Return dict with per-user and aggregated metrics
        pass
    
    def evaluate_beyond_accuracy(self, recs):
        # TODO: Implement ILD, Coverage, Novelty, Serendipity
        pass
    
    def evaluate_hallucination(self, recs, explanations=None):
        # TODO: Detect item hallucination (items outside catalog)
        # TODO: If explanations provided, detect explanation hallucination
        # Return hallucination rate and affected users
        pass
    
    def evaluate_fairness(self, recs, user_groups):
        # TODO: Compute group fairness gap and provider Gini
        pass
    
    def evaluate_all(self, recs, explanations=None, user_groups=None):
        # TODO: Run all evaluations and return consolidated report
        pass
    
    def compare_systems(self, recs_a, recs_b, names=('System A', 'System B')):
        # TODO: Produce side-by-side comparison report
        pass


# TODO: Instantiate and test
# framework = EvaluationFramework(item_catalog, ground_truth, N_ITEMS)
# report = framework.evaluate_all(gen_recs, gen_explanations, user_groups)
# framework.compare_systems(gen_recs, disc_recs, ('Generative', 'Discriminative'))

### Exercise 2: Implement Calibration-Based Evaluation

Calibration measures whether the category distribution in recommendations matches the user's historical preference distribution. Implement calibration using KL-divergence.

In [ ]:
def calibration_score(rec_list, user_preference_dist, catalog, categories, k=10):
    """Compute recommendation calibration using KL-divergence.
    
    TODO:
    1. Compute the category distribution of the rec list:
       q(c) = (# items from category c in rec_list) / k
    2. Compute KL(p || q) where p is the user's historical category distribution:
       KL(p || q) = sum_c p(c) * log(p(c) / q(c))
    3. Handle zero probabilities with Laplace smoothing:
       q_smooth(c) = (count(c) + epsilon) / (k + epsilon * |C|)
    4. Return the calibration score (lower KL = better calibration)
    """
    # TODO: Implement
    pass


# TODO: Compute user preference distributions from user_history_cats
# TODO: Evaluate calibration for both generative and discriminative systems
# TODO: Plot the distribution of calibration scores across users
# TODO: Compare: which system is better calibrated to user preferences?

### Exercise 3: Implement Temporal Consistency Evaluation

Evaluate whether generative recommendations are temporally consistent: given similar user states at different time steps, does the system produce similar recommendations?

In [ ]:
def temporal_consistency(rec_lists_t1, rec_lists_t2, k=10):
    """Measure temporal consistency between two sets of recommendations.
    
    TODO:
    1. For each user, compute Jaccard similarity between recs at t1 and t2:
       J(L_t1, L_t2) = |L_t1 \cap L_t2| / |L_t1 \cup L_t2|
    2. Compute rank correlation (Kendall tau) for overlapping items
    3. Aggregate across users: mean, std, and distribution
    4. A generative system with high temperature may have LOW consistency
       (same input -> different outputs), which can be a feature or a bug
    """
    # TODO: Implement
    pass


# TODO: Simulate two runs of the generative system (different random seeds)
# TODO: Compute temporal consistency
# TODO: Compare with discriminative system (should be perfectly consistent)
# TODO: Discuss: when is inconsistency desirable (exploration) vs harmful (trust)?

### Exercise 4: Design a Hallucination Severity Scoring System

Not all hallucinations are equally harmful. Design a weighted scoring system.

In [ ]:
def hallucination_severity_score(rec_list, explanations, catalog, n_items,
                                  user_history):
    """Compute a weighted hallucination severity score.
    
    TODO:
    1. For each recommendation, check for:
       a) Item hallucination (item not in catalog): severity = 1.0
       b) Category mismatch (explanation mentions wrong category): severity = 0.5
       c) False popularity claim ("best seller" for unpopular item): severity = 0.3
       d) History fabrication ("you bought X" but user never did): severity = 0.7
    
    2. Weight by position (hallucination at rank 1 is worse than rank 10):
       position_weight = 1 / log2(rank + 1)
    
    3. Compute total severity score:
       S = sum(severity_i * position_weight_i) / sum(position_weight_i)
    
    4. Return overall score in [0, 1] where 0 = no hallucination
    """
    # TODO: Implement
    pass


# TODO: Apply to the generative recommendations
# TODO: Identify the most severe hallucination cases
# TODO: Plot severity distribution and analyze patterns
# TODO: Propose mitigation strategies for the most common hallucination types

### Exercise 5: Statistical Significance Testing

Implement proper statistical testing to determine if differences between systems are significant.

In [ ]:
def paired_bootstrap_test(metric_a, metric_b, n_bootstrap=10000, rng=None):
    """Paired bootstrap test for comparing two systems.
    
    TODO:
    1. Compute the observed difference: delta = mean(A) - mean(B)
    2. For each bootstrap iteration:
       a) Sample users with replacement
       b) Compute delta_boot = mean(A_boot) - mean(B_boot)
    3. Compute p-value: fraction of bootstrap deltas that are <= 0
       (if we hypothesize A > B)
    4. Compute 95% confidence interval for the difference
    5. Return: observed delta, p-value, CI, and whether result is significant
       at alpha = 0.05
    """
    # TODO: Implement
    pass


# TODO: Test significance of NDCG difference between generative and discriminative
# TODO: Test significance of diversity difference
# TODO: Report results with proper statistical language
# TODO: Discuss: what sample size is needed for reliable A/B testing?

## Summary

### The Five Pillars of Generative Recommendation Evaluation

| Pillar | Key Metrics | Unique to GenRec? |
|--------|------------|-------------------|
| **Accuracy** | Precision@K, NDCG@K | No (standard) |
| **Beyond-Accuracy** | ILD, Novelty, Serendipity, Coverage | Partially (GenRec tends to be more diverse) |
| **Text Quality** | BLEU, ROUGE-L, BERTScore | Yes (explanation evaluation) |
| **Safety** | Hallucination rate, Explanation faithfulness | Yes (item/attribute/explanation hallucination) |
| **Fairness** | Group NDCG gap, Provider Gini, Language bias | Partially (language bias is GenRec-specific) |

### Key Formulas

| Metric | Formula |
|--------|--------|
| **ILD** | $\frac{2}{K(K-1)} \sum_{i \neq j} \text{dist}(i,j)$ |
| **Novelty** | $\frac{1}{K} \sum_{i} -\log_2 p(i)$ |
| **Serendipity** | $\frac{1}{K} \sum_{i} \mathbb{1}[i \in \text{Rel}] \cdot (1 - p(i))$ |
| **Gini** | $\frac{\sum_i \sum_j |x_i - x_j|}{2n \sum_i x_i}$ |
| **Cohen's $\kappa$** | $\frac{p_o - p_e}{1 - p_e}$ |

### Key Takeaways

1. **Multi-dimensional evaluation is essential** -- accuracy alone is insufficient for generative rec
2. **Hallucination is the critical new failure mode** -- must be detected and quantified at both item and explanation levels
3. **Fairness requires both exposure analysis and language audit** -- generative systems can amplify bias through language
4. **Human evaluation remains irreplaceable** -- automated metrics approximate but do not replace human judgment
5. **Generative and discriminative systems win in different scenarios** -- the choice is context-dependent, not absolute
6. **Statistical rigor matters** -- always report confidence intervals and significance tests when comparing systems